# Multi-Agent Financial Analysis System

AAI-520 Final Team Project

## 1. Project Overview and GitHub Repository

This notebook is the canonical project entry point. It demonstrates the reusable research workflow implemented under `src/` without redefining application logic in notebook cells.

## 2. Setup and Configuration

This section locates the repository, imports the public workflow interface, and sets the ticker. It does not contact Yahoo Finance or Ollama; those calls begin in the end-to-end section.

In [ ]:
import os
import subprocess
import time

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running in Colab. Installing Ollama in this VM, nothing touches your own machine.")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(5)  # give the server a moment to come up before pulling
    colab_model = os.getenv("OLLAMA_MODEL", "llama3.2")
    subprocess.run(["ollama", "pull", colab_model], check=True)
    print(f"Ollama is running in this Colab session with {colab_model} pulled.")
else:
    print("Not running in Colab. Assuming a local Ollama service is already up (ollama serve).")

In [ ]:
import os
import sys
from pathlib import Path

from IPython.display import JSON, Markdown, display


def find_project_root(start: Path) -> Path:
    """Find the repository whether Jupyter starts at its root or notebooks/."""
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src").is_dir() and (candidate / "requirements.txt").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the project repository.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.graph import build_research_workflow
from src.reporting import render_console_summary

workflow = build_research_workflow(project_root=PROJECT_ROOT)
TICKER = os.getenv("DEFAULT_TICKER", "AAPL").strip().upper()

print(f"Project root: {PROJECT_ROOT}")
print(f"Research ticker: {TICKER}")

Project root: /Users/brandonwirgau/Projects/AAI/520/Final Project
Research ticker: AAPL


## 3. Agent Design and Shared State

`ResearchWorkflow` coordinates planner, evaluator, synthesis, and memory components. Each stage adds a typed artifact to `ResearchState`, allowing the notebook and tests to inspect the same results.

## 4. Agent Functions, Data Sources, and Tool Use

The planner chooses from the registered tools. The executor invokes provider-independent tools, currently backed by the Yahoo Finance adapter and the NewsAPI-backed news tool, and records failures without stopping unrelated tool calls.

Adding a quick note here since this is my piece: `news` is now registered in `src/tools/registry.py` alongside the Yahoo Finance tools, pointing at `run_news_agent`. Same `Callable[[str], dict[str, Any]]` shape as everything else, so the planner and executor treat it like any other tool, no special casing needed. The actual retrieval and prompt chain live in `src/tools/news_tools.py` and `src/workflows/news_pipeline.py`, demoed live below in section 5.

## 5. Workflow 1 — Prompt Chaining

The implemented research chain passes structured artifacts through planning, collection, deterministic validation, reflection, synthesis, report validation, and memory curation. The news-specific ingest-to-summary chain below is the explicit Prompt Chaining pattern the assignment asks for, ingest, preprocess, classify, extract, summarize, each stage its own function so the output is inspectable at every step.

In [ ]:
from src.tools.news_tools import get_company_news

articles = get_company_news(TICKER, company_name="Apple")

print(f"Pulled {len(articles)} deduped articles for {TICKER}")
for a in articles[:3]:
    print(f"- {a['title']} ({a['publisher']}, {a['date']})")

Pulled 10 deduped articles for AAPL
- Apple CEO John Ternus Talks iPhone Duo Design, Siri AI, and Becoming CEO (MacRumors, 2026-09-21T23:12:27Z)
- Apple M6チップ搭載のMac miniのCPUベンチマークは、Single-CoreスコアはApple Siliconの中で最も高く、Multi-CoreスコアでもM1 Ultraに迫るスコアに。 (Applech2.com, 2026-09-21T22:59:20Z)
- Apple AirPods Pro 3 $169 + 2 Years AppleCare+ — Costco via DoorDash [YMMV] (Slickdeals.net, 2026-09-21T22:57:42Z)


In [ ]:
from src.workflows.news_pipeline import ingest, preprocess

ingested = ingest(articles)
preprocessed = preprocess(ingested)

print(f"Ingested: {len(ingested)} articles")
print(f"After preprocess (deduped, cleaned): {len(preprocessed)} articles")

Ingested: 10 articles
After preprocess (deduped, cleaned): 10 articles


Classify, extract, and summarize call the local LLM, same `ask_llm` / Ollama setup the Planner and Evaluator use, so no separate provider to configure. I didn't have Ollama running when I generated this version of the notebook, so the cell below ran for real but the model calls fell back to their safe defaults (OTHER category, neutral sentiment, no summary), that's the fallback I built in so one failed call doesn't take down the whole chain. Rerun this cell with Ollama up and `llama3.2` pulled to get the real classification and summary.

In [ ]:
from src.workflows.news_pipeline import classify, extract, summarize

classified = classify(preprocessed)
extracted = extract(classified)
news_summary = summarize(TICKER, extracted)

for a in classified[:3]:
    print(f"- [{a['category']}] {a['title']}")

print("\nSummary:")
print(news_summary)

- [OTHER] Apple CEO John Ternus Talks iPhone Duo Design, Siri AI, and Becoming CEO
- [OTHER] Apple M6チップ搭載のMac miniのCPUベンチマークは、Single-CoreスコアはApple Siliconの中で最も高く、Multi-CoreスコアでもM1 Ultraに迫るスコアに。
- [OTHER] Apple AirPods Pro 3 $169 + 2 Years AppleCare+ — Costco via DoorDash [YMMV]

Summary:
Unable to generate a news summary for AAPL.


## 6. Workflow 2 — Routing

Tool selection is allow-listed against the registry before execution. Specialist content routing can be added behind the existing router interface without changing this notebook entry point.

Flagging this honestly since I reviewed the PR this came from: allow-listing tool names isn't the same thing as the Routing pattern the assignment wants, sending content to the right specialist based on what it is. `src/agents/router.py` is still an empty stub. Still open, not mine to build unilaterally since it touches how the whole graph decides what runs.

## 7. Workflow 3 — Evaluator–Optimizer

The evaluator combines deterministic checks with an LLM quality reflection, then validates the synthesized report for coverage and unsafe language. The optimizer module is scaffolded for a future feedback-driven revision loop.

## 8. Memory and Learning Across Runs

After a successful run, the memory curator stores a short timestamped lesson. The planner can retrieve recent notes for the same symbol, but current tool data always remains the source of market evidence.

## 9. End-to-End Investment Research Example

The next cell is the project's live execution point. It requires a running Ollama service with the configured model and network access for Yahoo Finance. Change `TICKER` above, then run this section.

In [ ]:
result = workflow.run(TICKER, progress=print)

1/7 Planning the research run
2/7 Collecting market and financial evidence
Running tool: cash_flow
Running tool: company_info
Running tool: financials
Running tool: price_data
3/7 Validating tool observations
4/7 Reflecting on evidence quality
5/7 Synthesizing the research report
6/7 Validating the research report
7/7 Saving lessons for a future run


This run predates my `news` tool getting registered, so it only shows the four Yahoo Finance tools above, it was already baked into this notebook from Brandon's branch before I merged. Once this gets re-run with `news` in the registry, `Running tool: news` shows up in that list too and `result["observations"]["news"]` carries the sourced summary from the prompt chain. Didn't want to overwrite a real successful run just to add one more tool name to a log line.

### Inspect Structured Workflow Artifacts

In [ ]:
print(render_console_summary(result))
display(JSON({
    "plan": result["plan"],
    "validation": result["validation"],
    "reflection": result["reflection"],
    "report_validation": result["report_validation"],
    "memory_entry": result.get("memory_entry"),
}))

FINAL RESEARCH REPORT

Apple Inc. (AAPL)
----------------------------------------------------------------------

1. COMPANY OVERVIEW
   Sector:              Technology
   Industry:            Consumer Electronics
   Country:             United States
   Current Price:       $338.98
   Market Cap:          $4.947T
   Enterprise Value:    $4.969T

2. PRICE PERFORMANCE
   One-Year Return:     32.86%
   Annualized Volatility: 24.56%
   Maximum Drawdown:    -13.80%

3. VALUATION
   Trailing P/E:        38.92
   Forward P/E:         35.35
   Price/Sales (TTM):   10.60
   Profit Margin:       27.62%
   Operating Margin:    32.62%
   Return on Equity:    148.75%
   Beta:                1.085

4. FINANCIAL PERFORMANCE (2025)
   Revenue:             $416.161B
   Operating Income:    $133.050B
   Net Income:          $112.010B
   EBITDA:              $144.748B
   Diluted EPS:         $7.46
   EBITDA / Net Income: 1.292x

5. CASH FLOW (2025)
   Operating Cash Flow: $111.482B
   Free Cash Flow:    

<IPython.core.display.JSON object>

### Final Research Report

In [ ]:
display(Markdown(result["report"]))

<IPython.core.display.Markdown object>

## 10. Evaluation, Limitations, and Conclusions

The workflow surfaces deterministic validation issues and model-generated quality feedback for inspection. Current limitations include reliance on one implemented market-data provider, a local model, and a single-pass report. News retrieval and the prompt-chaining pipeline are implemented and registered as a tool; content-based routing and an iterative revision loop are still open. Outputs are research support, not personalized investment advice.